<div style="display:flex; align-items:flex-start; gap:2rem; padding:1rem 0;">

  <div style="flex-shrink:0;">
    <img src="https://cdn.oreillystatic.com/images/sitewide-headers/oreilly_logo_mark_red.svg"
         style="height:36px; display:block; margin-bottom:0.75rem;"/>
    <img src="https://i.imgur.com/ITL8dZE.jpeg"
         style="width:260px; border-radius:4px; box-shadow:0 2px 8px rgba(0,0,0,0.15); display:block;"/>
  </div>

  <div style="flex:1; font-family:sans-serif;">
    <h1 style="font-size:1.6rem; font-weight:600; margin:0 0 0.25rem 0;">
      AI, ML and GenAI in the Lakehouse
    </h1>
    <p style="margin:0 0 1rem 0; color:#555; font-size:0.95rem;">
      <strong>Chapter 06-01 &mdash; Lending Club Feature Store Example</strong><br/>
      Author: Bennie Haelen &nbsp;&bull;&nbsp; Date: 02-09-2025
    </p>
    <p style="margin:0 0 0.75rem 0; color:#333; font-size:0.9rem;">
      Loads the Lending Club dataset, engineers features into a Unity Catalog feature table
      using <code>FeatureEngineeringClient</code>, trains a Random Forest classifier for
      loan default prediction, and logs the model with full feature lineage to Unity Catalog
      via MLflow.
    </p>
    <table style="border-collapse:collapse; font-size:0.85rem; color:#333; width:100%;">
      <tr style="background:#f5f5f5;">
        <td style="padding:4px 10px; font-weight:600; white-space:nowrap;">Section</td>
        <td style="padding:4px 10px; font-weight:600;">Steps</td>
      </tr>
      <tr>
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>0 &mdash; Setup</strong></td>
        <td style="padding:4px 10px;">Install package &bull; Import libraries &bull; Set constants &bull; Initialize <code>FeatureEngineeringClient</code></td>
      </tr>
      <tr style="background:#f9f9f9;">
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>1 &mdash; Load &amp; preprocess</strong></td>
        <td style="padding:4px 10px;">Load dataset &bull; Generate unique ID &bull; Clean percentage fields</td>
      </tr>
      <tr>
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>2 &mdash; Feature engineering</strong></td>
        <td style="padding:4px 10px;">Select initial features &bull; Create schema &bull; Create feature table &bull; Select additional features &bull; Merge additional features</td>
      </tr>
      <tr style="background:#f9f9f9;">
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>3 &mdash; Training dataset</strong></td>
        <td style="padding:4px 10px;">Create label DataFrame &bull; Define <code>FeatureLookup</code>s &bull; Create training set &bull; Load &amp; prepare pandas DataFrame &bull; Impute missing values</td>
      </tr>
      <tr>
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>4 &mdash; Train model</strong></td>
        <td style="padding:4px 10px;">Train/test split &bull; Train Random Forest &bull; Predictions &amp; accuracy &bull; Infer signature</td>
      </tr>
      <tr style="background:#f9f9f9;">
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>5 &mdash; Visualizations</strong></td>
        <td style="padding:4px 10px;">Feature importance &bull; Confusion matrix &bull; Target distribution &bull; ROC curve &bull; Precision-recall curve</td>
      </tr>
      <tr>
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>6 &mdash; Log &amp; register</strong></td>
        <td style="padding:4px 10px;">Define Unity Catalog model name &bull; Log model with feature lineage via <code>fe.log_model()</code></td>
      </tr>
    </table>
  </div>

</div>

# Section 0: Setup and Prerequisites

## 0-1: Install the databricks-feature-engineering package
The `databricks-feature-engineering` package is pre-installed in Databricks Runtime 13.3 LTS ML
and above. If you are running on a non-ML runtime, uncomment and run the cell below.

In [0]:
# Uncomment if running on a non-ML Databricks Runtime:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

## 0-2: Import libraries
Note the import path: `databricks.feature_engineering` replaces the deprecated
`databricks.feature_store`. The `FeatureEngineeringClient` is the correct client
for all Unity Catalog feature table operations.

In [0]:
# Feature Engineering in Unity Catalog -- replaces the deprecated databricks.feature_store
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

# PySpark
from pyspark.sql.functions import col, monotonically_increasing_id, regexp_replace, trim

# scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# MLflow
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 0-3: Set constants
Run the shared constants notebook if it exists in your workspace, or set the values
inline below. The inline cell is the authoritative fallback -- update it to match
your Unity Catalog environment before running.

In [0]:
# Update these values to match your Unity Catalog environment.

CATALOG_NAME    = "book_ai_ml_lakehouse"
FEATURE_STORE_DB = "feature_store_db"

## 0-4: Initialize the FeatureEngineeringClient and MLflow registry
`FeatureEngineeringClient` is the Unity Catalog-native client. It replaces the
deprecated `FeatureStoreClient` for all workspaces enabled for Unity Catalog.

Setting the MLflow registry URI to `databricks-uc` ensures that all model
registrations go to Unity Catalog rather than the legacy Workspace Model Registry.

In [0]:
# Initialize the Feature Engineering client for Unity Catalog
fe = FeatureEngineeringClient()

# Point MLflow at Unity Catalog for all model registrations in this notebook.
# This must be set before any mlflow.register_model or fe.log_model call.
mlflow.set_registry_uri("databricks-uc")

# Section 1: Load and Preprocess the Lending Club Dataset

## 1-1: Load the dataset

In [0]:
# Load the Lending Club dataset from Databricks' built-in public datasets.
# The dataset is stored in Parquet format and contains loan application records
# including borrower demographics, loan terms, and repayment outcomes.
lending_club_path = "/databricks-datasets/samples/lending_club/parquet"
df = spark.read.parquet(lending_club_path)

print(f"Loaded {df.count():,} records with {len(df.columns)} columns.")

## 1-2: Add a unique ID to the dataset
Feature tables in Unity Catalog require a primary key. The Lending Club dataset does
not include a natural unique identifier, so we generate one using
`monotonically_increasing_id()`. This function produces a unique 64-bit integer
per row and is safe to use as a surrogate primary key for this example.

In [0]:
# Add a surrogate primary key.
# monotonically_increasing_id() guarantees uniqueness within the DataFrame
# but values are not sequential -- they reflect Spark partition boundaries.
df = df.withColumn("unique_id", monotonically_increasing_id())

## 1-3: Clean percentage fields
The `int_rate` and `revol_util` columns are stored as percentage strings
(e.g., `'13.56%'`). We strip the `%` symbol and cast to double so these
columns can be used as numeric features.

In [0]:
# Remove the '%' character and cast both fields to double.
# trim() removes any leading/trailing whitespace before the regex replace.
df = df.withColumn("int_rate",   regexp_replace(trim(col("int_rate")),   "%", "").cast("double"))
df = df.withColumn("revol_util", regexp_replace(trim(col("revol_util")), "%", "").cast("double"))

# Section 2: Feature Engineering

## 2-1: Select the initial subset of features
We use a two-pass approach to simulate iterative feature development, which is
common in real-world ML pipelines. The first pass selects core loan and
borrower attributes. A second pass will merge in credit history features.

In [0]:
# Initial feature set: core loan terms and borrower financial profile.
# unique_id is included as the primary key -- it will not be used as a model feature.
initial_features_df = df.select(
    col("unique_id"),    # Primary key
    col("loan_amnt"),    # Loan amount requested by the borrower
    col("int_rate"),     # Interest rate (cleaned to numeric above)
    col("annual_inc"),   # Borrower's annual income
    col("dti"),          # Debt-to-income ratio
    col("revol_util"),   # Revolving credit utilization rate (cleaned above)
    col("open_acc"),     # Number of open credit lines
)

## 2-2: Ensure the feature store schema exists

In [0]:
%sql
-- Create the schema if it does not already exist.
-- Unity Catalog uses a three-level namespace: catalog.schema.table
CREATE SCHEMA IF NOT EXISTS book_ai_ml_lakehouse.feature_store_db;

## 2-3: Define the feature table name

In [0]:
# Construct the fully qualified feature table name: catalog.schema.table
FEATURE_TABLE_NAME = "lending_club_loan"
feature_table_name = f"{CATALOG_NAME}.{FEATURE_STORE_DB}.{FEATURE_TABLE_NAME}"
print(f"Feature table: {feature_table_name}")

## 2-4: Drop the table if it exists (for clean re-runs)
Dropping via Spark SQL is the cleanest approach -- it removes both the Delta
table and its Unity Catalog metadata in one operation. The `IF EXISTS` clause
makes the cell safe to run on a fresh workspace where the table does not yet exist.

In [0]:
# Drop the existing table so the notebook can be re-run from scratch.
spark.sql(f"DROP TABLE IF EXISTS {feature_table_name}")
print(f"Dropped table (if it existed): {feature_table_name}")

## 2-5: Create the feature table
In Unity Catalog, any Delta table with a primary key constraint is automatically
a feature table -- no separate registration step is required. `fe.create_table()`
creates the underlying Delta table, sets `unique_id` as a NOT NULL primary key,
and writes the initial feature data in a single operation.

Note: primary key constraints in Unity Catalog are informational and are not
enforced at write time. They signal the join key to the Feature Engineering
platform for training set construction and inference lookups.

In [0]:
# Create the feature table and write the initial feature set.
# fe.create_table() creates the Delta table in Unity Catalog, sets the primary
# key constraint on unique_id, and persists the DataFrame in one call.
fe.create_table(
    name=feature_table_name,
    primary_keys=["unique_id"],
    df=initial_features_df,
    description="Lending Club loan features for default prediction -- initial feature set",
)

print(f"Feature table created: {feature_table_name}")

## 2-6: Select the additional subset of features
The second pass adds credit history columns. These provide deeper insight into
borrower risk behavior and are expected to improve model performance.

In [0]:
# Additional credit history features.
# We select directly from df -- no RDD round-trip is needed.
additional_features_df = df.select(
    col("unique_id"),     # Primary key -- required for the merge
    col("delinq_2yrs"),   # 30+ day delinquencies in the last 2 years
    col("pub_rec"),       # Derogatory public records (bankruptcies, liens)
    col("total_acc"),     # Total number of credit lines
    col("mort_acc"),      # Number of mortgage accounts
)

## 2-7: Merge the additional features into the feature table
Using `mode="merge"` joins the new columns into the existing table on `unique_id`.
Rows already present are updated; new rows are inserted. Existing columns
from the first write are preserved, demonstrating how the feature table
supports incremental, non-destructive feature development.

In [0]:
# Merge the additional credit history features into the existing feature table.
# mode='merge' preserves all existing rows and columns, adding the new columns
# alongside them. This is the standard pattern for iterative feature development.
fe.write_table(
    name=feature_table_name,
    df=additional_features_df,
    mode="merge",
)

print(f"Additional features merged into: {feature_table_name}")

# Section 3: Build the Training Dataset Using FeatureEngineeringClient
Rather than manually reading the feature table and joining it to labels, we use
`fe.create_training_set()` with `FeatureLookup` objects. This approach is
preferred because it records the feature lineage in the training set object,
which `fe.log_model()` later uses to link the registered model back to its
source feature table in Unity Catalog.

## 3-1: Create the label DataFrame
The label DataFrame must contain the lookup key (`unique_id`) and the target
variable (`defaulted`). The Feature Engineering client joins features from
the feature table onto this DataFrame using `unique_id`.

In [0]:
# Build the label DataFrame.
# A loan is considered defaulted if loan_status == 'Charged Off'.
# unique_id is the join key that the FeatureLookup will use to attach features.
label_df = df.select(
    col("unique_id"),
    (col("loan_status") == "Charged Off").alias("defaulted"),
)

## 3-2: Define feature lookups
A `FeatureLookup` specifies which columns to retrieve from a feature table and
which key to join on. We retrieve all 10 engineered features from the table
in a single lookup, joining on `unique_id`.

In [0]:
# Define which features to retrieve from the feature table at training time.
# These same lookups will be reused by fe.log_model() so the registered model
# knows how to retrieve features automatically at inference time.
feature_lookups = [
    FeatureLookup(
        table_name=feature_table_name,
        feature_names=[
            "loan_amnt",
            "int_rate",
            "annual_inc",
            "dti",
            "revol_util",
            "open_acc",
            "delinq_2yrs",
            "pub_rec",
            "total_acc",
            "mort_acc",
        ],
        lookup_key="unique_id",
    )
]

## 3-3: Create the training set
`fe.create_training_set()` joins the label DataFrame with the feature table
using the FeatureLookup definitions. The returned `training_set` object carries
the feature lineage metadata -- keep this object, as it is passed to
`fe.log_model()` in Section 6 to link the model to its source features.

In [0]:
# Create the training set.
# exclude_columns removes unique_id from the feature matrix -- it is a join key,
# not a predictive feature, and must be excluded so the model is not trained on it.
training_set = fe.create_training_set(
    df=label_df,
    feature_lookups=feature_lookups,
    label="defaulted",
    exclude_columns=["unique_id"],
)

## 3-4: Load the training DataFrame and convert to pandas
`training_set.load_df()` materializes the joined Spark DataFrame. We convert
to pandas for compatibility with scikit-learn.

In [0]:
# Materialize the training DataFrame and convert to pandas.
training_pd = training_set.load_df().toPandas()

print(f"Training DataFrame shape: {training_pd.shape}")
print(f"Columns: {list(training_pd.columns)}")

## 3-5: Drop rows with missing target values

In [0]:
# Remove any rows where the target label is null.
# scikit-learn cannot train on undefined labels.
training_pd = training_pd.dropna(subset=["defaulted"])
print(f"Shape after dropping null labels: {training_pd.shape}")

## 3-6: Separate features from the target variable

In [0]:
# Separate the feature matrix (X) from the target vector (y).
# We drop 'defaulted' from X and retain its column names for the
# feature importance plot in Section 5.
X = training_pd.drop(columns=["defaulted"])
y = training_pd["defaulted"]

# Preserve feature column names for the importance plot.
feature_cols = X.columns.tolist()
print(f"Feature columns: {feature_cols}")

## 3-7: Ensure all features are numeric

In [0]:
# Coerce all columns to numeric types.
# Any values that cannot be converted (e.g., residual string tokens) become NaN
# and will be handled by the imputer in the next step.
X = X.apply(pd.to_numeric, errors="coerce")

## 3-8: Impute missing values

In [0]:
# Replace NaN values with the column mean.
# Mean imputation is a simple, practical choice for this numeric feature set.
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

print(f"Feature matrix shape after imputation: {X_imputed.shape}")

# Section 4: Train the Model

## 4-1: Split into train and test datasets

In [0]:
# 80/20 train/test split with a fixed random seed for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]:,}")
print(f"Test samples:     {X_test.shape[0]:,}")

## 4-2: Train the Random Forest classifier

In [0]:
# Train a Random Forest with 100 estimators.
# y_train is cast to int because scikit-learn classifiers expect integer labels
# and the 'defaulted' column is boolean after the label DataFrame was created.
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train.astype(int))

print("Model training complete.")

## 4-3: Make predictions and evaluate accuracy

In [0]:
# Cast y_test to int for consistency with the classifier's integer output.
y_test  = y_test.astype(int)
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # Positive-class probabilities for ROC/PR curves

accuracy = accuracy_score(y_test, y_pred)

print(f"Model accuracy: {accuracy:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Not Defaulted", "Defaulted"]))

## 4-4: Infer the model signature
The MLflow model signature records the input and output schema. It is attached to
the logged model so that serving infrastructure can validate requests at inference time.

In [0]:
# Infer the signature from the test feature matrix and predictions.
# This captures input column types and the output type, which MLflow stores
# alongside the model artifact.
signature = infer_signature(X_test, y_pred)

# Section 5: Model Visualizations

## 5-1: Feature importance plot

In [0]:
# Build a sorted DataFrame of feature importances from the trained Random Forest.
importance_df = (
    pd.DataFrame({"Feature": feature_cols, "Importance": model.feature_importances_})
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
sns.barplot(data=importance_df, x="Importance", y="Feature", palette="Blues_d")
plt.title("Feature importance -- Lending Club default prediction", fontsize=14)
plt.xlabel("Mean decrease in impurity", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.show()

## 5-2: Confusion matrix

In [0]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted: not defaulted", "Predicted: defaulted"],
    yticklabels=["Actual: not defaulted",    "Actual: defaulted"],
)
plt.title("Confusion matrix", fontsize=14)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True negatives:  {tn:,}  |  False positives: {fp:,}")
print(f"False negatives: {fn:,}  |  True positives:  {tp:,}")

## 5-3: Target variable distribution

In [0]:
plt.figure(figsize=(7, 5))
sns.set_style("whitegrid")
ax = sns.countplot(
    x="defaulted",
    data=training_pd,
    palette="coolwarm",
    edgecolor="black",
)
ax.set_title("Distribution of loan defaults", fontsize=14)
ax.set_xlabel("Loan status (0 = fully paid, 1 = defaulted)", fontsize=12)
ax.set_ylabel("Number of loans", fontsize=12)

for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center", va="bottom", fontsize=10,
    )

sns.despine()
plt.tight_layout()
plt.show()

## 5-4: ROC curve

In [0]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random classifier")
plt.xlabel("False positive rate", fontsize=12)
plt.ylabel("True positive rate", fontsize=12)
plt.title("ROC curve -- Lending Club default prediction", fontsize=14)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 5-5: Precision-recall curve

In [0]:
precision, recall, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, color="coral", lw=2, label=f"AUC = {pr_auc:.3f}")
plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precision", fontsize=12)
plt.title("Precision-recall curve -- Lending Club default prediction", fontsize=14)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Section 6: Log and Register the Model in Unity Catalog
`fe.log_model()` is the single call that logs the model artifact, records the
feature lineage from the `training_set`, and registers the model in Unity Catalog.
This replaces the old pattern of calling `mlflow.sklearn.log_model()`,
`mlflow.register_model()`, and `fs.log_model()` separately.

## 6-1: Define the Unity Catalog model name

In [0]:
# The registered model name follows the Unity Catalog three-level namespace.
model_name       = "lending_club_default_model"
unity_model_name = f"{CATALOG_NAME}.{FEATURE_STORE_DB}.{model_name}"

print(f"Model will be registered as: {unity_model_name}")

## 6-2: Log the model with feature lineage using fe.log_model()
Passing `training_set` to `fe.log_model()` is what creates the Unity Catalog
lineage link. The registered model will show which feature table it was trained
on in Catalog Explorer, and at batch inference time `fe.score_batch()` will
automatically retrieve feature values from the feature table using the same lookups.

In [0]:
with mlflow.start_run() as run:

    # Log evaluation metrics for experiment tracking.
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc",  roc_auc)
    mlflow.log_metric("pr_auc",   pr_auc)

    # Log and register the model.
    # fe.log_model() handles three things in one call:
    #   1. Logs the sklearn model artifact under the current MLflow run.
    #   2. Attaches the training_set so Unity Catalog can record feature lineage.
    #   3. Registers the model at unity_model_name, creating a new version if
    #      the registered model already exists.
    fe.log_model(
        model=model,
        artifact_path="model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name=unity_model_name,
    )

print(f"Run ID: {run.info.run_id}")
print(f"Model registered: {unity_model_name}")
print("Done -- model and feature lineage logged to Unity Catalog.")

# End of Notebook

After running this notebook you can verify the results in Databricks:

1. **Feature table** -- open Catalog Explorer, navigate to
   `book_ai_ml_lakehouse > feature_store_db > lending_club_loan`.
   You will see all 10 feature columns, the primary key constraint on `unique_id`,
   and the lineage tab showing this notebook as a producer.

2. **Registered model** -- navigate to
   `book_ai_ml_lakehouse > feature_store_db > lending_club_default_model`.
   The model detail page shows the version, metrics, and a lineage graph
   linking back to `lending_club_loan`.